In [1]:
import pandas as pd
import numpy as np
import requests  # <-- On utilise requests à la place de urllib

def get_movies_by_year(year):
    """
    Fonction robuste pour récupérer les films de Wikipédia.
    Cherche automatiquement les tableaux contenant le bon entête.
    """
    link = f"https://en.wikipedia.org/wiki/List_of_American_films_of_{year}"
    try:
        # --- L'ASTUCE ANTI-403 EST ICI ---
        # On se déguise en vrai navigateur Google Chrome sur Windows
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        # On télécharge la page web avec notre faux passeport
        response = requests.get(link, headers=headers)
        
        if response.status_code != 200:
            print(f"⚠️ Erreur d'accès à la page pour {year} (Code HTTP: {response.status_code})")
            return pd.DataFrame()

        # On donne le texte de la page à Pandas pour qu'il trouve les tableaux
        tables = pd.read_html(response.text, match='Cast and crew')
        # ---------------------------------
        
        if not tables:
            print(f"⚠️ Aucun tableau de films trouvé pour {year}")
            return pd.DataFrame()
            
        # On fusionne tous les tableaux trouvés sur cette page
        df = pd.concat(tables, ignore_index=True)
        
        # Harmonisation du nom de la colonne du titre
        if 'Title' in df.columns:
            df = df.rename(columns={'Title': 'movie_title'})
            
        # On vérifie qu'on a bien nos deux colonnes vitales
        if 'movie_title' in df.columns and 'Cast and crew' in df.columns:
            df = df[['movie_title', 'Cast and crew']]
            df['year'] = year
            return df
        else:
            return pd.DataFrame()
            
    except Exception as e:
        print(f"⚠️ Erreur lors du scan de l'année {year} : {e}")
        return pd.DataFrame()

# --- BOUCLE POUR RÉCUPÉRER DE 2017 À 2026 ---
all_new_movies = []
for year in range(2017, 2027):
    print(f"Extraction des films de l'année : {year}...")
    yearly_df = get_movies_by_year(year)
    
    if not yearly_df.empty:
        # On supprime les lignes vides bizarres de Wikipédia
        yearly_df = yearly_df.dropna(subset=['movie_title', 'Cast and crew'])
        all_new_movies.append(yearly_df)
        print(f"✅ {year} : {len(yearly_df)} films trouvés !")

if len(all_new_movies) == 0:
    print("\n❌ ERREUR CRITIQUE : Aucun film n'a été téléchargé. Impossible de continuer.")
else:
    # Fusion de tous les nouveaux films
    df_2017_2026 = pd.concat(all_new_movies, ignore_index=True)

    # Extraction des acteurs et réalisateurs
    def get_director(x):
        if " (director)" in str(x):
            return str(x).split(" (director)")[0]
        elif " (directors)" in str(x):
            return str(x).split(" (directors)")[0]
        else:
            return "unknown"

    df_2017_2026['director_name'] = df_2017_2026['Cast and crew'].map(lambda x: get_director(x))

    def get_actor1(x):
        return ((str(x).split("screenplay); ")[-1]).split(", ")[0])
        
    def get_actor2(x):
        if len((str(x).split("screenplay); ")[-1]).split(", ")) >= 2:
            return ((str(x).split("screenplay); ")[-1]).split(", ")[1])
        else:
            return "unknown"
            
    def get_actor3(x):
        if len((str(x).split("screenplay); ")[-1]).split(", ")) >= 3:
            return ((str(x).split("screenplay); ")[-1]).split(", ")[2])
        else:
            return "unknown"

    df_2017_2026['actor_1_name'] = df_2017_2026['Cast and crew'].map(lambda x: get_actor1(x))
    df_2017_2026['actor_2_name'] = df_2017_2026['Cast and crew'].map(lambda x: get_actor2(x))
    df_2017_2026['actor_3_name'] = df_2017_2026['Cast and crew'].map(lambda x: get_actor3(x))

    # Création de la colonne 'comb' pour la recommandation
    df_2017_2026['comb'] = df_2017_2026['actor_1_name'] + ' ' + df_2017_2026['actor_2_name'] + ' ' + df_2017_2026['actor_3_name'] + ' ' + df_2017_2026['director_name']
    
    # Nettoyage final : mise en minuscule des titres
    df_2017_2026['movie_title'] = df_2017_2026['movie_title'].str.lower()

    # Sauvegarde finale
    df_2017_2026.to_csv('movies_2017_2026.csv', index=False)
    print(f"\n🎉 SUCCÈS ! Fichier 'movies_2017_2026.csv' créé avec {len(df_2017_2026)} films !")

Extraction des films de l'année : 2017...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11840\221281122.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, match='Cast and crew')


✅ 2017 : 242 films trouvés !
Extraction des films de l'année : 2018...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11840\221281122.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, match='Cast and crew')


✅ 2018 : 251 films trouvés !
Extraction des films de l'année : 2019...


...

✅ 2024 : 494 films trouvés !
Extraction des films de l'année : 2025...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11840\221281122.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, match='Cast and crew')


✅ 2025 : 493 films trouvés !
Extraction des films de l'année : 2026...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11840\221281122.py:26: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text, match='Cast and crew')


✅ 2026 : 322 films trouvés !

🎉 SUCCÈS ! Fichier 'movies_2017_2026.csv' créé avec 3367 films !
